# Import libraries

In [1]:
import getpass
import os
import chromadb
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
#from langchain.vectorstores import Chroma
#from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

/Users/jeremykrick/miniconda3/envs/BAH_RAG/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Verify API keys

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

# Load document

In [3]:
# Document loader
file_path = (
    "/Users/jeremykrick/Downloads/BUPERSINST 1900.8.pdf"
)

loader = PyPDFLoader(file_path)
pages = loader.load()

In [4]:
print(f"{pages[1].metadata}\n")
print(pages[1].page_content)

{'producer': 'Adobe Acrobat Pro (32-bit) 24 Paper Capture Plug-in', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2025-03-31T12:23:21-05:00', 'author': 'PC4073', 'company': 'Saztec Philippines, Inc.', 'contenttypeid': '0x010100998ED7DC9E1E764EB54EFD225770C6C9', 'mediaserviceimagetags': '', 'moddate': '2025-03-31T13:03:54-05:00', 'sourcemodified': '', 'title': 'BUPERSINST INSTRUCTION 1900', 'source': '/Users/jeremykrick/Downloads/BUPERSINST 1900.8.pdf', 'total_pages': 46, 'page': 1, 'page_label': '2'}

BUPERSINST 1900.8F 
27 Mar 2025 
2 
3. Scope and Applicability
a. Thi
s instruction applies to all active duty and Navy Reserve Service members and the
total work force responsible for delivery of personnel and pay services and includes, but is not 
limited to: 
(1) Commanding officers (CO), commanders, heads of activities, and officers in charge
(OIC) with responsibility of executing the certification of Navy military personnel and pay 
functions. 
(2) Command administrator

# Split document (Chunking)


In [5]:
# Text-structured splitting (organized into hierarchical units such as paragraphs, sentences, and words)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
# Create a list of documents
documents = loader.load_and_split(text_splitter=text_splitter)

# Embeddings

In [ ]:
# Initialize with an embedding model
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = InMemoryVectorStore(embedding=embeddings_model)

vector_store.add_documents(documents=documents)

# Context retrieval and call LLM

In [7]:
#import langchain_core.vectorstores.base
query = "What is lump-sum leave?"
docs = vector_store.similarity_search(query, k=2) # k = # of chunks to retrieve

In [8]:
# Define a system prompt that tells the model how to use the retrieved context
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Context: {context}:"""

# combine the documents into a single string
context_text = "".join(doc.page_content for doc in docs)

# Populate the system prompt with the retrieved context
system_prompt_fmt = system_prompt.format(context=context_text)


# Llama-3.1-8b

In [9]:
# Create a model
Llama_model = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)
# Generate a response
ai_msg = Llama_model.invoke([SystemMessage(content=system_prompt_fmt),
                          HumanMessage(content=query)])
print(ai_msg.content)

Lump-sum leave is a payment made to a Service member for accrued leave time that they have not used before separation from the military. It is a one-time payment for the unused leave balance. The payment is usually made in addition to regular pay.


# GPT-4o

In [10]:
# Create a model
GPT4o_model = ChatOpenAI(model="gpt-4o", temperature=0)
# Generate a response
ai_msg = GPT4o_model.invoke([SystemMessage(content=system_prompt_fmt),
                          HumanMessage(content=query)])
print(ai_msg.content)

Lump-sum leave is a payment made to a service member for unused leave days when they separate from military service. Instead of taking the leave as time off, the service member receives a monetary compensation for the accrued leave days. The number of days for which payment is received is recorded, but the amount of the payment itself is not shown.


# Gradio UI

In [11]:
rag_model = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

rag_system_prompt = """You are an assistant for question-answering tasks.\nUse the provided context to answer the question concisely.\nIf the answer is not in the context, say you do not know.\nContext: {context}"""

def format_docs(docs):
    if not docs:
        return "No relevant context retrieved."
    formatted = []
    for idx, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source") or doc.metadata.get("page", "unknown")
        formatted.append(f"Chunk {idx} (source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

def generate_response(message, history):
    docs = vector_store.similarity_search(message, k=3)
    context_text = format_docs(docs)
    prompt = rag_system_prompt.format(context=context_text)
    response = rag_model.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=message),
    ])
    history = history + [[message, response.content]]
    return history, context_text, ""

def clear_state():
    return [], "", ""

with gr.Blocks() as rag_demo:
    gr.Markdown("### Groq RAG Chatbot")
    chatbot = gr.Chatbot(height=400)
    question = gr.Textbox(label="Ask a question", placeholder="Ask about the policy…", lines=1)
    context_box = gr.Textbox(label="Retrieved context", lines=10)
    submit = gr.Button("Submit")
    clear = gr.Button("Clear conversation")

    submit.click(generate_response, inputs=[question, chatbot], outputs=[chatbot, context_box, question])
    question.submit(generate_response, inputs=[question, chatbot], outputs=[chatbot, context_box, question])
    clear.click(clear_state, outputs=[chatbot, context_box, question])

rag_demo.launch()


/var/folders/1n/mc3sw9jx2ms_nq89095jvg140000gn/T/ipykernel_24236/2049630555.py:30: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
